### Set-up

In [ ]:
## setting paths and all
from pathlib import Path  ### switching to pathlib path-handling instead of the os - should work consistently between HPC and local machine
import sys

# current working directory of the notebook / script
cwd = Path.cwd()

# assume "scripts" folder is one level up from cwd
project_root = cwd.parent.resolve()

if str(project_root) not in sys.path:
    print("Adding to sys.path:", project_root)
    sys.path.append(str(project_root)) # add root to Python path (as a string) for finding scripts modules further


#2
# 0. loading libraries (could be removed after all modules are loaded from the scripts)
import glob
import tifffile
import numpy as np
import matplotlib.pyplot as plt
from skimage.io import imread
from skimage.color import rgb2gray
from skimage.color import label2rgb

from skimage.exposure import rescale_intensity
from skimage.transform import resize

from cellpose import models
from stardist.models import StarDist3D 

# loading scripts
from scripts.pre_processing import preprocess_3d_image ### pre-processing function (normalisation + optional downsampling)
from scripts.segment_3d import segment_with_stardist, save_segmentation_results ### StarDist3D (3d_demo) segmentation model - light and relatively quick to run


In [ ]:
def load_multichannel_images(input_folder, channel_map):
    """
    Loads multi-channel 3D images from a folder and returns a structured list 
    with filename and channel-specific volumes.
    
    Args:
        input_folder (str or Path): Folder with .tif files
        channel_map (dict): Mapping like {"nuclei": 0, "cytoplasm": 1, "membrane": 2}

    Returns:
        list of dict: [
            {
                "filename": str,  # original TIFF filename
                "channels": {
                    "nuclei": array(Z,Y,X),
                    "cytoplasm": array(Z,Y,X),
                    ...
                }
            },
            ...
        ]
    """
    input_folder = Path(input_folder)
    files = sorted(input_folder.glob("*.tif"))
    
    if not files:
        raise FileNotFoundError("No .tif files found in the folder.")

    all_volumes = []
    for f in files:
        img = tifffile.imread(f)  # could be (Z,Y,X,C) or (C,Z,Y,X)
        
        if img.ndim != 4:
            raise ValueError(f"Unexpected image shape: {img.shape}. Expected 4D (Z,Y,X,C) or (C,Z,Y,X).")

        # identify channels axis and move it to last
        if img.shape[-1] <= 10:
            zyx_channels = img  # already in (Z,Y,X,C)
        else:
            # guess channel axis (the one with size < 10)
            channel_axis = np.argmin(img.shape)
            if img.shape[channel_axis] < 10:
                zyx_channels = np.moveaxis(img, channel_axis, -1) # moving the channel axis to the last position
            else:
                raise ValueError(f"Cannot determine channel axis for shape {img.shape}")

        # map channels to names
        channels_dict = {}
        for name, idx in channel_map.items():
            if idx >= zyx_channels.shape[-1]:
                raise IndexError(f"Channel index {idx} for '{name}' out of range in image {f.name}")
            channels_dict[name] = zyx_channels[..., idx]

        # append structured entry with filename and channels (per image)
        all_volumes.append({
            "filename": f.stem, 
            "channels": channels_dict
        })

    return all_volumes

In [ ]:
###### configuration ######

#input_folder = project_root.parent.resolve() / "input_data" / "Images_BBBC050" / "test" / "Images" # put the directory to your input data, relative to the project folder
                                                                                                   # project_root.parent() derives directory above the project folder root
#input_folder = project_root.parent.resolve() / "input_data" / "BBBC035" / "BBBC035_v1_dataset" / "01" 

input_folder = project_root.parent.resolve() / "input_data" / "2017_07_21_Tom20" / "AICS-11-part13" 


In [ ]:
# 1. load images as multichannel dictionary

channel_map = {
    "nucleus": 2,
    "cytoplasm": 0,
    "mitochondria": 1
}

all_volumes = load_multichannel_images(input_folder, channel_map)

# preview loaded images: first image [0] as example
print(all_volumes[0]["filename"])       # e.g., "sample01.tif"
print(all_volumes[0]["channels"].keys()) # dict_keys(['nucleus', 'cytoplasm', 'mitochondria'])
print(all_volumes[0]["channels"]["nucleus"].shape)  # (Z, Y, X)

#### MAIN: per-channel preprocessing, segmentation, and quantification

In [ ]:
### configs ###
#OUTPUT_DIR = project_root.parent / "output_data" / "Images_BBBC050" / "test" / "Images" 
OUTPUT_DIR = project_root.parent / "output_data" / "2017_07_21_Tom20" / "AICS-11-part13" 
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

### params ###
num_test_volumes = 10  # number of volumes from `frames` to segment
use_model = "stardist"  # "cellpose" or "stardist"
diameter = None        # or specify an estimate, e.g., 20
channels = [0, 0]      # grayscale input for Cellpose

### pre-processing params ###
downsize_factor = 0.5   # scaling factor or 1 to keep original                                 TODO: make sure that downsampling is proportionate? (e.g. if the original size is not 1:1)
per_slice_norm = False        # True = normalize per-slice, False = normalize whole stack


##### a) nucleus

In [ ]:
# 2. nucleus segmentation

results_dict = {}

# loop over loaded images for segmentation
for v in all_volumes:
    file_name = v['filename']  # preserve original file name
    
    # --- nuclei segmentation ---
    nuclei_volume = v["channels"]["nucleus"]
    nuclei_norm = preprocess_3d_image(nuclei_volume, downsize_factor=downsize_factor, per_slice=per_slice_norm)
    
    if use_model == "stardist":
        nuclei_mask = segment_with_stardist(nuclei_norm)
    elif use_model == "cellpose":
        nuclei_mask = segment_with_cellpose(nuclei_norm, diameter=diameter, channels=channels)
    
    # saving masks in a dictionary for future use
    if file_name not in results_dict:
        results_dict[file_name] = {}    # initialize dictionary for this file if it doesn't exist

    results_dict[file_name]["nuclei_mask"] = nuclei_mask 
    
    # save nuclei mask
    save_segmentation_results(
        nuclei_norm, 
        nuclei_mask, 
        output_root=OUTPUT_DIR, 
        experiment_label=f"{file_name}_nuclei",
        save_overlay=True
    )


In [ ]:
# (optional): visualise the example nucleus segmentation result using Napari (have not checked on myriad)

%gui qt
import napari

viewer = napari.Viewer(ndisplay=3)
viewer.add_image(nuclei_norm, name='Raw Image', rendering='mip', colormap='gray') ### adding the original 3d grey-scale image
viewer.add_labels(nuclei_mask, name='Segmentation Mask') ### adding a created mask overlay

##### b) cytoplasm

In [ ]:
# cytoplasm segemtnation function

from skimage.segmentation import watershed
from skimage.filters import gaussian
import numpy as np
from scipy import ndimage as ndi

def segment_cytoplasm(nuclei_mask, cyto_channel, mode="membrane", sigma=1.0,
                      min_signal=0.05, membrane_threshold=0.2):
    """
    Segment cytoplasm using nuclei as seeds and cytoplasmic/membrane channel as guidance.

    Parameters:
    ----------
    nuclei_mask : ndarray
        Labeled nuclei segmentation (3D).
    cyto_channel : ndarray
        Cytoplasmic or membrane channel (3D).
    mode : str
        "membrane" for boundary-based segmentation (membrane marker).
        "intensity" for intensity-based segmentation (cytoplasmic marker).
    sigma : float
        Gaussian smoothing for the guidance image.
    min_signal : float
        Minimum normalized intensity for cytoplasm mask (for intensity mode).
    membrane_threshold : float
        Threshold for membrane signal to act as stopping boundary.

    Returns:
    -------
    cytoplasm_labels : ndarray
        Labeled cytoplasm mask.
    """
    
    # normalize channel to 0-1
    channel = preprocess_3d_image(cyto_channel, downsize_factor, per_slice=per_slice_norm)
    
    # smooth channel
    smoothed = gaussian(channel, sigma=sigma)
    
    if mode == "membrane":
        # binary mask of membrane
        membrane_mask = smoothed > membrane_threshold
        distance = ndi.distance_transform_edt(~membrane_mask)
        cytoplasm_labels = watershed(-distance, markers=nuclei_mask, mask=~membrane_mask)
    
    elif mode == "intensity":
        # cytoplasmic regions to include
        cytoplasm_mask = smoothed > min_signal
        inverted = -smoothed
        cytoplasm_labels = watershed(inverted, markers=nuclei_mask, mask=cytoplasm_mask)
    
    else:
        raise ValueError("Invalid mode. Choose 'membrane' or 'intensity'.")
    
    return cytoplasm_labels


In [ ]:
from skimage.segmentation import watershed
from skimage.filters import gaussian
import numpy as np
from scipy import ndimage as ndi

def segment_cytoplasm(nuclei_mask, cyto_channel, mode="membrane", sigma=1.0,
                      min_signal=0.05, membrane_threshold=0.2):
    """
    Segment cytoplasm using nuclei as seeds and cytoplasmic/membrane channel as guidance.

    Parameters:
    ----------
    nuclei_mask : ndarray
        Labeled nuclei segmentation (3D).
    cyto_channel : ndarray
        Cytoplasmic or membrane channel (3D).
    mode : str
        "membrane" for boundary-based segmentation (membrane marker).
        "intensity" for intensity-based segmentation (cytoplasmic marker).
    sigma : float
        Gaussian smoothing for the guidance image.
    min_signal : float
        Minimum normalized intensity for cytoplasm mask (for intensity mode).
    membrane_threshold : float
        Threshold for membrane signal to act as stopping boundary.

    Returns:
    -------
    cytoplasm_labels : ndarray
        Labeled cytoplasm mask.
    """
    
    # Normalize channel to 0-1
    channel = (cyto_channel - cyto_channel.min()) / (
        cyto_channel.max() - cyto_channel.min()
    )
    
    # Smooth channel
    smoothed = gaussian(channel, sigma=sigma)
    
    if mode == "membrane":
        # Binary mask of membrane
        membrane_mask = smoothed > membrane_threshold
        distance = ndi.distance_transform_edt(~membrane_mask)
        cytoplasm_labels = watershed(-distance, markers=nuclei_mask, mask=~membrane_mask)
    
    elif mode == "intensity":
        # Cytoplasmic regions to include
        cytoplasm_mask = smoothed > min_signal
        inverted = -smoothed
        cytoplasm_labels = watershed(inverted, markers=nuclei_mask, mask=cytoplasm_mask)
    
    else:
        raise ValueError("Invalid mode. Choose 'membrane' or 'intensity'.")
    
    return cytoplasm_labels


In [ ]:
# 3. cytoplasm segmentation

# loop over loaded images for segmentation
for v in all_volumes:
    file_name = v['filename']  # preserve original file name

    # get the corresponding nuclear mask
    if file_name not in results_dict:
        raise KeyError(f"Nuclear mask for {file_name} not found in previous results.")
    nuclei_mask = results_dict[file_name]["nuclei_mask"]

    # --- cytoplasm segmentation (watershed) ---
    cytoplasm_volume = v["channels"]['cytoplasm']
    cytoplasm_norm = preprocess_3d_image(
            cytoplasm_volume, 
            downsize_factor=downsize_factor, 
            per_slice=per_slice_norm
        )    
    cytoplasm_mask = segment_cytoplasm(
        cytoplasm_norm, 
        nuclei_mask, 
        mode="membrane"  # could also be "cytoskeleton" for alternative approach
    )
    
    # saving masks in a dictionary for future use
    if file_name not in results_dict:
        results_dict[file_name] = {}    # initialize dictionary for this file if it doesn't exist
    
    results_dict[file_name]["cytoplasm_mask"] = cytoplasm_mask


    save_segmentation_results(
        cytoplasm_norm, 
        cytoplasm_mask, 
        output_root=OUTPUT_DIR, 
        experiment_label=f"{file_name}_cytoplasm",
        save_overlay=True
    )

In [ ]:
# (optional): visualise the example cytoplasm segmentation result using Napari (have not checked on myriad)

%gui qt
import napari

viewer = napari.Viewer(ndisplay=3)
viewer.add_image(cytoplasm_norm, name='Raw Image', rendering='mip', colormap='gray') ### adding the original 3d grey-scale image
viewer.add_labels(cytoplasm_mask, name='Segmentation Mask') ### adding a created mask overlay
viewer.add_labels(nuclei_mask, name='Nuc Segmentation Mask') ### adding a created mask overlay

##### c) organelle